In [1]:
using DualNumbers
using ForwardDiff
using LinearAlgebra
using Quadmath

In [2]:
toRadians::Float128 = pi/180
numberOfAtoms::Int64 = 6
masses::Vector{Float128} = Float128.([12.00000000, 15.99491463, 1.00782503223, 1.00782503223, 1.00782503223, 1.00782503223])
valenceCoordinates = Float128.([1.42167426, 0.95631445, 1.09061209, 1.09061209, 1.09061209, 108.10210467*toRadians, 110.28974516*toRadians, 110.28974516*toRadians, 110.28974516*toRadians, 0.0, 0.0, pi/3])

12-element Vector{Float128}:
 1.42167426000000007846324479032773525e+00
 9.56314450000000038087932807684410363e-01
 1.09061209000000003399577508389484137e+00
 1.09061209000000003399577508389484137e+00
 1.09061209000000003399577508389484137e+00
 1.88673765482703823842586014038349470e+00
 1.92491918422748019513219346619590753e+00
 1.92491918422748019513219346619590753e+00
 1.92491918422748019513219346619590753e+00
 0.00000000000000000000000000000000000e+00
 0.00000000000000000000000000000000000e+00
 1.04719755119659763131778618117095903e+00

In [54]:
function defineMolecularFrameXYZ(valence)
    # Stores type to prevent type conversion problems with duals
    T = eltype(valence) 
    molecularFrameXYZ = zeros(T, numberOfAtoms, 3)
    # C at origin
    # CO defines z-axis
    molecularFrameXYZ[2, 3] = valence[1]
    # OH defines x-axis
    molecularFrameXYZ[3, 1] = valence[2]*sin(valence[6])
    molecularFrameXYZ[3, 3] = valence[1]-valence[2]*cos(valence[6])
    # CH1
    molecularFrameXYZ[4, 1] = valence[3]*sin(valence[7])*cos(valence[12] - sqrt(2)*valence[11]/3)
    molecularFrameXYZ[4, 2] = valence[3]*sin(valence[7])*sin(valence[12] - sqrt(2)*valence[11]/3)
    molecularFrameXYZ[4, 3] = valence[3]*cos(valence[7])
    # CH2
    molecularFrameXYZ[5, 1] = valence[4]*sin(valence[8])*cos(4*pi/3 + valence[12] + valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[5, 2] = valence[4]*sin(valence[8])*sin(4*pi/3 + valence[12] + valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[5, 3] = valence[4]*cos(valence[8])
    # CH3
    molecularFrameXYZ[6, 1] = valence[5]*sin(valence[9])*cos(2*pi/3 + valence[12] - valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[6, 2] = valence[5]*sin(valence[9])*sin(2*pi/3 + valence[12] - valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[6, 3] = valence[5]*cos(valence[9])
    
    return molecularFrameXYZ
end

function transformToCOM(molecularFrame)
    COM = masses'*molecularFrame/sum(masses)
    frameCOM = molecularFrame .- COM
    return frameCOM
end

function defineCOMframe(valence)
    molecularFrame = defineMolecularFrameXYZ(valence)
    frameCOM = transformToCOM(molecularFrame)
    return frameCOM
end

function computeTMatrix(valence)
    frameCOM = defineCOMframe(valence)
    T = eltype(valence)
    tMatrix = zeros(T, numberOfAtoms*3, numberOfAtoms*3)
    for i in 1:6
        # Translational part
        tMatrix[i*3-2:i*3, 1:3] = Matrix(1I, 3, 3)
    
        # Rotational Part
        for j in 1:3
            unitVector = Matrix(1I, 3, 3)[j, :]
            tMatrix[i*3-2:i*3, 3 + j] = cross(unitVector, frameCOM[i, :])
        end
        # Vibrational part
        for j in 1:3
            cartesianValenceGradient = ForwardDiff.gradient(valence -> defineCOMframe(valence)[i, j], valence) 
            tMatrix[i*3-3+j, 7:end] = cartesianValenceGradient
        end
    end
    return tMatrix
end

function computeSMatrix(valence)
    tMatrix = computeTMatrix(valence)
    sMatrix = inv(tMatrix)
    return sMatrix
end

function computeGMatrix(valence)
    sMatrix = computeSMatrix(valence)
    T = eltype(valence)
    GMatrix = zeros(T, 3*numberOfAtoms, 3*numberOfAtoms)
    for i in 1:3*numberOfAtoms
        for j in 1:3*numberOfAtoms
            for n in 1:numberOfAtoms
                GMatrix[i, j] += dot(sMatrix[i, n*3-2:n*3], sMatrix[j, n*3-2:n*3])/masses[n]
            end          
        end
    end
    return GMatrix
end

computeGMatrix (generic function with 1 method)

In [37]:
tMatrix = computeTMatrix(valenceCoordinates)
sMatrix = inv(tMatrix)

18×18 Matrix{Float128}:
  3.74693047253039417261521578634412272e-01  …   1.50463276905252801019998276764447447e-36
 -4.42169318976253393645805236663173886e-35     -6.68191177523048911535134116787870148e-52
 -9.02779661431516806119989660586652805e-36      3.14687527020126127975680513789426709e-02
  7.22659690853622254195893694635890494e-35     -4.32761555494170821976187211998600616e-85
 -7.03396008590603549936443804937428505e-01      0.00000000000000000000000000000000000e+00
  1.36649295735455135715608418105062005e-34  …  -8.18318255855741179361432321644802356e-85
  0.00000000000000000000000000000000000e+00      0.00000000000000000000000000000000000e+00
 -1.18390754869659320019587636232877184e-34     -0.00000000000000000000000000000000000e+00
 -4.68975507191425525283364184342959052e-01      6.09289080858463851320834252792000281e-85
 -4.68975507191424965600879523303621575e-01     -2.92434179656194232985574958274269899e-85
  9.37951014382850864005900148339447466e-01  …  -3.467677819783451

In [41]:
@time GMatrix = computeGMatrix(valenceCoordinates)

  0.004644 seconds (4.59 k allocations: 795.844 KiB)


18×18 Matrix{Float128}:
  3.12244206044199514384601315528676833e-02  …  -4.18719579020158989663321863312234816e-38
  3.13856051875444840627353041583594801e-37     -1.05324293833676960713998793735113213e-35
  9.62546212590454041605344441861946753e-38      1.23873129997954746394831591136459459e-36
  2.28782711319181388464457232761059353e-36     -7.19692018272743900944682360186116848e-02
 -1.25264949198512647419497870504419878e-35     -3.84078579327131213481507923961182526e-33
  2.46863796297839244989324099680318721e-36  …  -1.31590385924362549633195657240402986e+00
  1.27570148259167784739226544270383418e-36     -7.62377726902653936598634255738817305e-37
 -7.52316384526264005099991383822237234e-35      6.91849406659907005662981872218765389e-34
 -1.50463276905252810871534156214736170e-36      1.55644027150270938109045953026931690e-02
 -1.50463276905252768016711314760182887e-36     -1.55644027150271000025026995619735006e-02
  6.01853107621011204079993107057789787e-36  …   6.191598104259282

In [43]:
GMatrix[18, 18]

1.63198069852602447241310294825574524e+00

In [33]:
GMatrix[18, 17]

-4.80892875767042428393724713562895819e-02

In [31]:
GMatrix[4, 4]

7.21631984797708460639297779067140935e-02

In [32]:
GMatrix[5, 5]

7.21631984797708460639297779067141175e-02

In [217]:
GMatrix[7, 7]

1.45853204375620977274330193446118726e-01

In [218]:
1/sum(masses)

3.12244206044199514384601315528676773e-02

In [219]:
GMatrix[1, 1]

3.12244206044199514384601315528676833e-02

In [161]:
GMatrix[2, 2]

3.12244206044199514384601315528676894e-02

In [30]:
GMatrix[3, 3]

3.12244206044199514384601315528676533e-02

In [13]:
sMatrix = computeSMatrix(valenceCoordinates)

18×18 Matrix{Float128}:
  3.74693047253039417261521578634412272e-01  …   1.50463276905252801019998276764447447e-36
 -4.42169318976253393645805236663173886e-35     -6.68191177523048911535134116787870148e-52
 -9.02779661431516806119989660586652805e-36      3.14687527020126127975680513789426709e-02
  7.22659690853622254195893694635890494e-35     -4.32761555494170821976187211998600616e-85
 -7.03396008590603549936443804937428505e-01      0.00000000000000000000000000000000000e+00
  1.36649295735455135715608418105062005e-34  …  -8.18318255855741179361432321644802356e-85
  0.00000000000000000000000000000000000e+00      0.00000000000000000000000000000000000e+00
 -1.18390754869659320019587636232877184e-34     -0.00000000000000000000000000000000000e+00
 -4.68975507191425525283364184342959052e-01      6.09289080858463851320834252792000281e-85
 -4.68975507191424965600879523303621575e-01     -2.92434179656194232985574958274269899e-85
  9.37951014382850864005900148339447466e-01  …  -3.467677819783451

In [53]:
Uterm1 = Float128(0.0)
for i in 1:numberOfAtoms
    for j in 1:3
        for k in 1:3
            Uterm1 += sMatrix[3 + j, 3*i - 3 + k]*(sMatrix[3 + j, 3*i - 3 + k] - sMatrix[3 + k, 3*i - 3 + j])/(8*masses[i])
        end
    end
end
Uterm1

2.00569581645338610824963188998765609e-01

In [ ]:
Uterm2 = Float128(0.0)
for i in 1:numberOfAtoms
    for j in 1:3
        for k in 1:3
            Uterm2 += sMatrix[3 + j, 3*i - 3 + k]*(sMatrix[3 + j, 3*i - 3 + k] - sMatrix[3 + k, 3*i - 3 + j])/(8*masses[i])
        end
    end
end


In [94]:
Dual(Float128(0.0), Float128(1.0))

0.00000000000000000000000000000000000e+00 + 1.00000000000000000000000000000000000e+00ɛ

In [79]:
typeof(Dual(2.0, 1.0))

Dual128 (alias for Dual{Float64})

In [75]:
dualpart(tan(Dual(2, 1)))

5.774399204041917

In [167]:
ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> x^2, x), Float128(2.0))

2.00000000000000000000000000000000000e+00

In [169]:
ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> x^3, x), x), Float128(2.0))

6.00000000000000000000000000000000000e+00